# Article Classifier — Exploration Notebook

**Cél:** Megérteni a `facebook/bart-large-mnli` zero-shot pipeline működését cikk-kategorizálásra.

Ez a notebook a projekt 1. napi mérföldköve — első működő prototípust hozunk létre, kipróbáljuk pár cikken, és látjuk, milyen eredményeket ad a modell.

## Tartalom

1. Környezet és modell betöltése
2. Egy egyszerű példa
3. Több cikk-kategória kipróbálása
4. Hosszú szöveg viselkedése (truncation challenge)
5. Tanulságok, következő lépések

## 1. Környezet és modell betöltése

A `transformers` library `pipeline` API-ját használjuk, ami a tokenizer + modell + post-processing lépéseket egyetlen hívásban összefogja.

Apple Silicon Mac-en az MPS (Metal Performance Shaders) device-ot használjuk a gyorsabb inference-hez.

In [ ]:
import torch
from transformers import pipeline

# Eszköz választás: MPS Apple Silicon-on, CUDA NVIDIA-n, különben CPU
if torch.backends.mps.is_available():
    device = "mps"
elif torch.cuda.is_available():
    device = "cuda"
else:
    device = "cpu"

print(f"Eszköz: {device}")
print(f"PyTorch verzió: {torch.__version__}")

In [ ]:
# Modell betöltése
# Első futtatáskor letölti a modellt (~1.6 GB) a HuggingFace cache-be (~/.cache/huggingface/)
MODEL_NAME = "facebook/bart-large-mnli"

classifier = pipeline(
    task="zero-shot-classification",
    model=MODEL_NAME,
    device=device,
)

print(f"✅ Modell betöltve: {MODEL_NAME}")

## 2. Egy egyszerű példa

Próbáljuk ki egy nyilvánvaló cikk-szövegen. A `candidate_labels` listával adjuk meg, mely kategóriákba sorolható a szöveg.

In [ ]:
article = (
    "Apple unveiled its new M5 chip today, claiming a 40% performance boost "
    "over the previous generation while consuming 30% less power. The chip "
    "will debut in next month's MacBook Pro lineup."
)

candidate_labels = ["technology", "sports", "politics", "business", "health"]

result = classifier(article, candidate_labels)

print(f"Predikció: {result['labels'][0]} (confidence: {result['scores'][0]:.3f})")
print()
print("Összes pontszám:")
for label, score in zip(result['labels'], result['scores']):
    print(f"  {label:15s} → {score:.3f}")

## 3. Több cikk kategorizálása

Demonstráljuk, hogy a modell különböző domain-eken is működik.

In [ ]:
test_articles = [
    {
        "id": 1,
        "text": "The Lakers defeated the Celtics 112-104 in overtime last night, with LeBron James scoring 35 points.",
        "expected": "sports",
    },
    {
        "id": 2,
        "text": "The Federal Reserve raised interest rates by 25 basis points, citing persistent inflation concerns.",
        "expected": "business",
    },
    {
        "id": 3,
        "text": "Researchers at MIT have developed a new battery technology that could double EV range.",
        "expected": "technology",
    },
    {
        "id": 4,
        "text": "The Senate passed a bipartisan infrastructure bill on Thursday, allocating $1.2 trillion to roads and bridges.",
        "expected": "politics",
    },
    {
        "id": 5,
        "text": "A new study suggests that consuming 30g of fiber daily reduces cardiovascular disease risk by 15%.",
        "expected": "health",
    },
]

candidate_labels = ["technology", "sports", "politics", "business", "health"]

for article in test_articles:
    result = classifier(article["text"], candidate_labels)
    predicted = result["labels"][0]
    confidence = result["scores"][0]
    correct = "✅" if predicted == article["expected"] else "❌"
    print(f"{correct} #{article['id']}: predicted={predicted:11s} (conf={confidence:.2f}) | expected={article['expected']}")
    print(f"   Text: {article['text'][:80]}...")
    print()

## 4. Hosszú szöveg viselkedése (truncation challenge)

A BART modell maximum kontextus-mérete **1024 token**. Ha hosszabb cikket adunk neki, a tokenizer levágja. Ez egy **fontos limitáció**, amit a végleges pipeline-ban kezelni kell.

Próbáljuk meg egy mesterségesen hosszú szöveggel:

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Egy tipikus rövid cikk
short_text = test_articles[0]["text"]
print(f"Rövid szöveg: {len(short_text)} karakter, {len(tokenizer.encode(short_text))} token")

# Mesterségesen hosszú szöveg (cikk * 50)
long_text = (test_articles[0]["text"] + " ") * 50
print(f"Hosszú szöveg: {len(long_text)} karakter, {len(tokenizer.encode(long_text))} token")
print(f"Limit: 1024 token → {'OK' if len(tokenizer.encode(long_text)) <= 1024 else 'TÚL HOSSZÚ, TRUNCATION KELL'}")

In [ ]:
# Ha ráengedjük a hosszú szövegre, a pipeline figyelmeztet és/vagy hibát dob.
# Megoldás: explicit truncation a tokenizerrel.

# 1) Nyers próbálkozás (csak demo céljából)
try:
    result = classifier(long_text, candidate_labels)
    print(f"Predikció: {result['labels'][0]} ({result['scores'][0]:.2f})")
except Exception as e:
    print(f"Hiba: {e}")

In [ ]:
# 2) Helyes megoldás: levágjuk a szöveget az első ~1000 tokenig (NLI hipotézisnek is kell hely)
MAX_INPUT_TOKENS = 900  # biztonsági ráhagyás a hipotézis-mondatra

def truncate_to_tokens(text: str, tokenizer, max_tokens: int) -> str:
    """Egyszerű truncation: visszaadja az első `max_tokens` tokennek megfelelő szöveget."""
    tokens = tokenizer.encode(text, add_special_tokens=False)
    if len(tokens) <= max_tokens:
        return text
    truncated_tokens = tokens[:max_tokens]
    return tokenizer.decode(truncated_tokens, skip_special_tokens=True)

long_truncated = truncate_to_tokens(long_text, tokenizer, MAX_INPUT_TOKENS)
print(f"Eredeti hossz: {len(tokenizer.encode(long_text))} token")
print(f"Truncated:     {len(tokenizer.encode(long_truncated))} token")

result = classifier(long_truncated, candidate_labels)
print(f"\nPredikció: {result['labels'][0]} ({result['scores'][0]:.2f})")

## 5. Tanulságok és következő lépések

### Mit tanultunk?

✅ A `bart-large-mnli` modell **out-of-the-box működik** zero-shot kategorizálásra.

✅ A confidence-eloszlás **informatív** — magas score (>0.8) általában helyes predikciót jelent, alacsony score (<0.5) viszont gyanús.

⚠️ **1024 token limit** — hosszú cikkeknél truncation kell. A jelenlegi egyszerű megoldás (első N token) elég, de éles használatban **chunking + averaging** javíthat a pontosságon.

⚠️ A modell **csak angolt** ért megbízhatóan.

### Következő lépések (2. nap)

1. **AG News dataset** letöltése a `datasets` library-vel
2. ~100 cikkből álló **eval sample** mentése `data/eval/ag_news_eval_v1.csv`-be
3. **Címkék verziózott konfigurációja** `data/prompts/labels_v1.json`-be
4. `src/pipeline.py` — production-grade wrapper a HF pipeline körül
5. **Kiértékelés** az eval seten: accuracy, per-class F1, confusion matrix